# Pipeline A — Step 2: Panel Fixed Effects OLS

**Model**: Panel regression dengan province dan year fixed effects.

```
twp90_avg_it = alpha_i + gamma_t + beta * X_it + epsilon_it
```

Di mana:
- `alpha_i` = province fixed effect (menangkap heterogeneity regional)
- `gamma_t` = year fixed effect (menangkap shock makro)
- `X_it` = fitur ekonomi (time-varying per provinsi)

**Diagnostics:**
- VIF (multicollinearity)
- Residual normality & autocorrelation
- Clustered standard errors
- Per-province evaluation

**Input:** `pipeline_A/output/A1_annual_panel.csv`

**Output:**
- `pipeline_A/output/A2_ols_summary.txt`
- `pipeline_A/output/A2_metrics.csv`
- `pipeline_A/output/A2_predictions.csv`
- `pipeline_A/output/A2_coefficients.csv`
- Visualization plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pipeline_A' / 'output' / 'A1_annual_panel.csv').exists():
            return p
    raise FileNotFoundError('Could not find A1_annual_panel.csv')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / 'pipeline_A' / 'output' / 'A1_annual_panel.csv'
output_dir = ROOT / 'pipeline_A' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_path)
print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Years: {sorted(df["tahun"].unique())}')
print(f'Provinces: {df["provinsi_id"].nunique()}')

In [ ]:
# === Train/Test Split ===
# Train: 2021-2023 (3 tahun x 31 prov = 93 obs)
# Test: 2024-2025 (2 tahun x 31 prov = 62 obs)

train = df[df['tahun'] <= 2023].copy()
test = df[df['tahun'] >= 2024].copy()

print(f'Train: {len(train)} obs (tahun {train["tahun"].min()}-{train["tahun"].max()})')
print(f'Test:  {len(test)} obs (tahun {test["tahun"].min()}-{test["tahun"].max()})')

# Drop baris dengan missing (dari lag = tahun pertama)
all_cols_needed = ['twp90_avg', 'x1_bi_rate_avg', 'x2_inflasi_avg',
                   'log_pdrb', 'x4_tpt', 'x5_internet',
                   'x8_ldr', 'x9_npl', 'x10_umkm']
train_clean = train.dropna(subset=all_cols_needed)
test_clean = test.dropna(subset=all_cols_needed)
print(f'\nAfter dropping NaN - Train: {len(train_clean)}, Test: {len(test_clean)}')

In [ ]:
# === MODEL 1: OLS with Province + Year Fixed Effects ===
# C(provinsi_id) = province fixed effects (alpha_i)
# C(tahun)       = year fixed effects (gamma_t)

formula = (
    'twp90_avg ~ '
    'x1_bi_rate_avg + x2_inflasi_avg + '
    'log_pdrb + x4_tpt + x5_internet + '
    'x8_ldr + x9_npl + x10_umkm + '
    'C(provinsi_id) + C(tahun)'
)

# Fit on TRAIN data with clustered standard errors by province
model_fe = smf.ols(formula, data=train_clean).fit(
    cov_type='cluster',
    cov_kwds={'groups': train_clean['provinsi_id']}
)

print('='*80)
print('MODEL: Panel OLS with Province + Year Fixed Effects')
print('Clustered standard errors by province')
print('='*80)
print(model_fe.summary())

# Save summary
with open(output_dir / 'A2_ols_summary.txt', 'w') as f:
    f.write(str(model_fe.summary()))
print(f'\nSaved: {output_dir / "A2_ols_summary.txt"}')

In [ ]:
# === VIF Check (multicollinearity) ===
# Hanya pada fitur kontinu (tanpa FE dummies)
vif_cols = ['x1_bi_rate_avg', 'x2_inflasi_avg', 'log_pdrb',
            'x4_tpt', 'x5_internet', 'x8_ldr', 'x9_npl', 'x10_umkm']
vif_data = train_clean[vif_cols].dropna()
vif_data = sm.add_constant(vif_data)

vif_results = pd.DataFrame({
    'feature': vif_data.columns,
    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]
}).sort_values('VIF', ascending=False)

print('=== VIF (Variance Inflation Factor) ===')
print('VIF > 10 = severe multicollinearity; VIF > 5 = moderate')
display(vif_results)

high_vif = vif_results[vif_results['VIF'] > 5]
if len(high_vif) > 1:  # exclude constant
    print(f'\n⚠️ {len(high_vif)-1} features with VIF > 5 detected')
else:
    print('\n✅ No severe multicollinearity detected')

In [ ]:
# === Predictions & Evaluation ===

# In-sample (train) predictions
train_clean = train_clean.copy()
train_clean['y_pred'] = model_fe.predict(train_clean)

# Out-of-sample (test) predictions
test_clean = test_clean.copy()
test_clean['y_pred'] = model_fe.predict(test_clean)

# Naive baseline: province mean from training set
prov_means = train_clean.groupby('provinsi_id')['twp90_avg'].mean()
test_clean['y_naive'] = test_clean['provinsi_id'].map(prov_means)

def calc_metrics(y_true, y_pred, name):
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_t, y_p = y_true[mask], y_pred[mask]
    return {
        'model': name,
        'rmse': np.sqrt(mean_squared_error(y_t, y_p)),
        'mae': mean_absolute_error(y_t, y_p),
        'r2': r2_score(y_t, y_p),
        'n': int(len(y_t)),
    }

m_train = calc_metrics(train_clean['twp90_avg'].values, train_clean['y_pred'].values, 'Panel FE OLS (Train)')
m_test = calc_metrics(test_clean['twp90_avg'].values, test_clean['y_pred'].values, 'Panel FE OLS (Test)')
m_naive = calc_metrics(test_clean['twp90_avg'].values, test_clean['y_naive'].values, 'Naive (Province Mean)')

print('='*70)
print(f'{"Model":<30} {"RMSE":>8} {"MAE":>8} {"R2":>8} {"N":>5}')
print('-'*70)
for m in [m_train, m_test, m_naive]:
    print(f'{m["model"]:<30} {m["rmse"]:>8.5f} {m["mae"]:>8.5f} {m["r2"]:>8.4f} {m["n"]:>5}')
print('='*70)

if m_test['rmse'] < m_naive['rmse']:
    improvement = (m_naive['rmse'] - m_test['rmse']) / m_naive['rmse'] * 100
    print(f'\n✅ Model beats naive baseline by {improvement:.1f}%')
else:
    print(f'\n🔴 Model WORSE than naive baseline')

gap = abs(m_train['rmse'] - m_test['rmse']) / m_test['rmse'] * 100
print(f'Overfitting gap: {gap:.1f}%')

# Save metrics
metrics_df = pd.DataFrame([m_train, m_test, m_naive])
metrics_df.to_csv(output_dir / 'A2_metrics.csv', index=False)
print(f'Saved: {output_dir / "A2_metrics.csv"}')

In [ ]:
# === Extract Interpretable Coefficients (tanpa FE dummies) ===
params = model_fe.params
pvalues = model_fe.pvalues
conf_int = model_fe.conf_int()

# Filter: hanya fitur ekonomi (bukan C(provinsi_id) dan C(tahun))
econ_features = [p for p in params.index if not p.startswith('C(') and p != 'Intercept']

coeff_df = pd.DataFrame({
    'feature': econ_features,
    'coefficient': [params[f] for f in econ_features],
    'p_value': [pvalues[f] for f in econ_features],
    'ci_lower': [conf_int.loc[f, 0] for f in econ_features],
    'ci_upper': [conf_int.loc[f, 1] for f in econ_features],
    'significant_5pct': [pvalues[f] < 0.05 for f in econ_features],
}).sort_values('p_value')

print('=== Economic Feature Coefficients ===')
display(coeff_df)

coeff_df.to_csv(output_dir / 'A2_coefficients.csv', index=False)
print(f'Saved: {output_dir / "A2_coefficients.csv"}')

In [ ]:
# === VISUALIZATION 1: Coefficient Plot ===
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(coeff_df))
colors = ['green' if s else 'gray' for s in coeff_df['significant_5pct']]
ax.barh(y_pos, coeff_df['coefficient'], color=colors, alpha=0.7)
ax.errorbar(coeff_df['coefficient'], y_pos,
            xerr=[coeff_df['coefficient'] - coeff_df['ci_lower'],
                  coeff_df['ci_upper'] - coeff_df['coefficient']],
            fmt='none', color='black', capsize=3)
ax.axvline(0, color='red', ls='--', alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(coeff_df['feature'])
ax.set_xlabel('Coefficient (with 95% CI)')
ax.set_title('Panel FE OLS — Economic Feature Coefficients\n(green = significant at 5%)')
fig.tight_layout()
fig.savefig(output_dir / 'A2_coefficient_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# === VISUALIZATION 2: Actual vs Predicted (Test) ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
ax = axes[0]
ax.scatter(test_clean['twp90_avg'], test_clean['y_pred'], alpha=0.6, s=40)
lims = [min(test_clean['twp90_avg'].min(), test_clean['y_pred'].min()),
        max(test_clean['twp90_avg'].max(), test_clean['y_pred'].max())]
ax.plot(lims, lims, 'r--', label='Perfect prediction')
ax.set_xlabel('Actual TWP90 (annual avg)')
ax.set_ylabel('Predicted TWP90')
ax.set_title('Actual vs Predicted (Test: 2024-2025)')
ax.legend()
ax.grid(True, alpha=0.3)

# Residual distribution
ax = axes[1]
residuals = test_clean['twp90_avg'] - test_clean['y_pred']
ax.hist(residuals, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', ls='--')
ax.set_title('Test Residual Distribution')
ax.set_xlabel('Residual (actual - predicted)')

fig.tight_layout()
fig.savefig(output_dir / 'A2_prediction_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === VISUALIZATION 3: Per-Province Prediction (Test Set) ===
fig, ax = plt.subplots(figsize=(14, 6))
prov_test = test_clean.groupby('nama_provinsi').agg(
    actual=('twp90_avg', 'mean'),
    predicted=('y_pred', 'mean'),
).sort_values('actual', ascending=True).reset_index()

x_pos = np.arange(len(prov_test))
width = 0.35
ax.barh(x_pos - width/2, prov_test['actual'], width, label='Actual', alpha=0.8)
ax.barh(x_pos + width/2, prov_test['predicted'], width, label='Predicted', alpha=0.8)
ax.set_yticks(x_pos)
ax.set_yticklabels(prov_test['nama_provinsi'], fontsize=7)
ax.set_xlabel('TWP90 (annual avg)')
ax.set_title('Per-Province: Actual vs Predicted (Test 2024-2025)')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'A2_per_province_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# === Save predictions ===
out_cols = ['provinsi_id', 'nama_provinsi', 'tahun', 'twp90_avg', 'y_pred', 'y_naive']
test_clean[out_cols].to_csv(output_dir / 'A2_predictions.csv', index=False)
print(f'Saved: {output_dir / "A2_predictions.csv"}')

In [ ]:
# === MODEL 2: Panel FE OLS with LAGGED target (autoregressive) ===
# Ini menambahkan twp90 tahun lalu sebagai prediktor — menangkap persistence

train_ar = train_clean.dropna(subset=['twp90_avg_lag1']).copy()
test_ar = test_clean.dropna(subset=['twp90_avg_lag1']).copy()

if len(train_ar) >= 30:
    formula_ar = (
        'twp90_avg ~ '
        'twp90_avg_lag1 + '
        'x1_bi_rate_avg + x2_inflasi_avg + '
        'log_pdrb + x4_tpt + x5_internet + '
        'x8_ldr + x9_npl + x10_umkm + '
        'C(provinsi_id) + C(tahun)'
    )

    model_ar = smf.ols(formula_ar, data=train_ar).fit(
        cov_type='cluster',
        cov_kwds={'groups': train_ar['provinsi_id']}
    )

    print('\n' + '='*80)
    print('MODEL 2: Panel FE OLS with Autoregressive Term (twp90_lag1)')
    print('='*80)

    # Evaluate AR model
    test_ar['y_pred_ar'] = model_ar.predict(test_ar)
    m_ar_test = calc_metrics(test_ar['twp90_avg'].values, test_ar['y_pred_ar'].values, 'Panel FE + AR(1) (Test)')

    print(f'\nAR Model Test: RMSE={m_ar_test["rmse"]:.5f}, MAE={m_ar_test["mae"]:.5f}, R2={m_ar_test["r2"]:.4f}')
    print(f'vs Base Model: RMSE={m_test["rmse"]:.5f}')

    # Print AR coefficient specifically
    ar_coef = model_ar.params.get('twp90_avg_lag1', None)
    ar_pval = model_ar.pvalues.get('twp90_avg_lag1', None)
    if ar_coef is not None:
        print(f'\ntwp90_lag1 coefficient: {ar_coef:.4f} (p={ar_pval:.4f})')
        if ar_pval < 0.05:
            print('  ✅ TWP90 shows significant persistence (autoregressive behavior)')
        else:
            print('  ⚠️ Lagged TWP90 not significant — limited autoregressive signal at annual level')
else:
    print(f'\nSkipping AR model: not enough observations after lag ({len(train_ar)} < 30)')

In [ ]:
# === FINAL SUMMARY ===
print('\n' + '='*80)
print('PIPELINE A — FINAL SUMMARY')
print('='*80)
print(f'\nData: {df["provinsi_id"].nunique()} provinces x {df["tahun"].nunique()} years = {len(df)} observations')
print(f'Model: Panel OLS with Province + Year Fixed Effects')
print(f'Train: {m_train["n"]} obs | Test: {m_test["n"]} obs')
print(f'\nTest RMSE: {m_test["rmse"]:.5f}')
print(f'Test R²: {m_test["r2"]:.4f}')
print(f'Naive RMSE: {m_naive["rmse"]:.5f}')

sig_features = coeff_df[coeff_df['significant_5pct']]
print(f'\nSignificant features (p < 0.05): {len(sig_features)}/{len(coeff_df)}')
for _, row in sig_features.iterrows():
    direction = '↑' if row['coefficient'] > 0 else '↓'
    print(f'  {direction} {row["feature"]}: coeff={row["coefficient"]:.5f}, p={row["p_value"]:.4f}')

print(f'\nInterpretasi: Koefisien menunjukkan arah ASOSIASI, BUKAN kausalitas.')
print(f'Untuk causal inference, diperlukan: instrumental variables, diff-in-diff, atau RCT.')
print('\nAll outputs saved to:', output_dir)